#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, regexp_replace
from pyspark.sql.window import Window

# Reading From Bronze

In [0]:
df = spark.table("salesdatalakehouse.bronze.erp_loc_a101_raw")

# Data Transformations

## Remove Invalid char from Customer ID

In [0]:
df = df.withColumn("cid", regexp_replace(col("cid"), "-", ""))

## Normalization

In [0]:
df = (
    df
    .withColumn("CNTRY",
                F.when(F.upper(trim(col("CNTRY"))).isin('US', 'USA'), "United States")
                .when(F.upper(trim(col("CNTRY"))) == 'DE', "Germany")
                .when((trim(col("CNTRY")) == '') | col("CNTRY").isNull(), "n/a")
                .otherwise(trim(col("CNTRY")))
    )
)

## Renaming Colmns

In [0]:
df = df.toDF(
    "customer_id",
    "country"
)

# Load Into Silver

In [0]:
df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable('salesdatalakehouse.silver.erp_loc_a101')